<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>Ejercicios II: Soluciones</h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>

---

<a id="indice"></a>
<h2><font color="#5E5E5E" size=5>Índice</font></h2>


* [9. Acceso condicional a datos](#section9)
    * [9.1 Selección condicional](#section91)          
    * [9.2 Filtrado por valor del índice](#section92) 
    * [9.3 Reemplazo condicional de valores](#section93)
* [10. Agrupaciones](#section10)
    * [10.1 Agrupación por categorías mediante `DataFrame.groupby()`](#section101)
    * [10.7 Agrupación por categorías externas. Grupos a partir de datos numéricos](#section103)        
* [11. Operaciones sobre grupos](#section11)        
    * [11.1 Agregación en grupos](#section111)     
    * [11.2 Filtrado por grupos](#section112)     
    * [11.3 Transformación por grupo](#section113)    
    * [11.4 `DataFrameGroupBy.apply()`](#section114)  
* [13. Combinación de DataFrames](#section13)
    * [13.2 Concatenación de *Series* y *DataFrames*](#section131)
    * [13.3 Fusión de *DataFrames*](#section132)
* [14. Índices de varios niveles](#section14)   
* [15. Tablas dinámicas y reestructuración](#section150)
    * [15.1 Creación de tablas dinámicas](#section151)
    * [15.2 Reestructuración de tablas](#section152)

---

Para esta serie de ejercicios se utilizará el conjunto de datos `fifa.csv`. 

In [ ]:
import pandas as pd
import numpy as np

# Lee un archivo .csv y utiliza el campo ID como índice.
df_fifa = pd.read_csv('./data/fifa19.csv', index_col=0).set_index('ID')

# Muestra la cabecera (5 primeras filas)
df_fifa.head()

A continuación, se almacenan en la lista `sel_columns` los nombres de las columnas de interés, y se eliminan las filas que tienen un `NaN` en alguna de ellas.

In [ ]:
# Lista con los nombres de las columnas de interés
sel_columns = ['Name', 'Age', 'Nationality', 'Overall', 
               'Potential', 'Club', 'Value', 'Wage', 'Position', 
               'Joined', 'Height', 'Weight', 'Release Clause']

df_fifa.dropna(subset=sel_columns, how='any', axis=0, inplace=True)

Se preprocesan las columnas que representan valores numéricos.

In [ ]:
feet_to_cm = 30.48 # Pies a centímetros
inch_to_cm = 2.54  # Pulgadas a centrímetros

# Toma un String con formato "pies'pulgadas" y devuelve un número con los metros
def imp_to_metric(imp_height):
    feet, inch = imp_height.split("'")                             # Separa pies y pulgadas 
    return (float(feet)*feet_to_cm + float(inch)*inch_to_cm)/100   # Convierte a float y calcula los metros

# Convierte toda la columna utilizando la función `imp_to_metric`.
df_fifa['Height'] = df_fifa['Height'].map(imp_to_metric)    
df_fifa['Weight'] = df_fifa['Weight'].map(lambda w: float(w[:-3])*0.453592) # Traduce a kilos la columna.
df_fifa['Value'] = df_fifa['Value'].map(lambda v: float(v[1:-1]))           # Convierte en numérico.
df_fifa['Wage'] = df_fifa['Wage'].map(lambda w: int(w[1:-1])*1000)          # Convierte a numérica la ganancia semanal.

Corrige el nombre del `Atlético de Madrid`.

In [ ]:
df_fifa['Club'] = df_fifa['Club'].replace('Atlético Madrid','Atlético de Madrid')

Crea el *DataFrame* `df_fifa_att` con las columnas que no están recogidas en `sel_columns`.

In [ ]:
df_fifa_att = df_fifa.drop(columns=sel_columns)
df_fifa_att.head()

Deja el el *DataFrame* `df_fifa` las columnas correspondientes a `sel_columns`.

In [ ]:
df_fifa = df_fifa[sel_columns].copy()
df_fifa.head()

<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section9"></a>
## <font color="#5E5E5E"> 9. Acceso condicional a datos </font>



<a id="section91"></a> 
### <font color="#5E5E5E">9.1 Selección condicional</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 1</font> 

Mostrar las filas correspondientes a los jugadores cuyo club es `Atlético de Madrid`. Utilizar selección condicional.

In [ ]:
df_fifa[df_fifa['Club']=='Atlético de Madrid']

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 2</font> 

Mostrar las filas correspondientes a los jugadores españoles cuyo club es `Atlético de Madrid`. Utilizar selección condicional.

In [ ]:
df_fifa[(df_fifa['Club']=='Atlético de Madrid') & (df_fifa['Nationality']=='Spain')]

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 3</font> 

Repetir el ejercicio anterior, pero utilizando el método `DataFrame.query()`.

In [ ]:
df_fifa.query("Club=='Atlético de Madrid' and Nationality=='Spain'")

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 4</font> 

Mostrar los juguadores españoles cuyo club es el Atlético de Madrid, y para los que la diferencia entre su potencial (`Potential`) y valoración global (`Overall`) es menor que cinco.

In [ ]:
df_fifa[(df_fifa['Club']=='Atlético de Madrid') & 
        (df_fifa['Nationality']=='Spain') & 
        (df_fifa['Potential']-df_fifa['Overall']<5)]

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 5</font> 

Mostrar los jugadores que militan en el Real Madrid, FC Barcelona o Atlético de Madrid. Utilizar el método `Series.isin()`.

In [ ]:
df_fifa[df_fifa['Club'].isin(['Real Madrid', 'FC Barcelona', 'Atlético de Madrid'])]

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 6</font> 

Mostrar los jugadores cuya altura está entre 1.70 y 1.90 cm. Utilizar  el método `Series.between()`.

In [ ]:
df_fifa[df_fifa['Height'].between(1.7, 1.9)]



<a id="section92"></a> 
### <font color="#5E5E5E">9.2 Filtrado por valor del índice</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 7</font> 

El *DataFrame* `df_fifa_att` contiene columnas con información sobre las características de los jugadores. Las características de los porteros se distinguen porque contienen las letras `GK` en su nombre. Mostrar las columnas correspondientes a estas características.

In [ ]:
df_fifa_att.filter(like='GK')


<a id="section93"></a> 
### <font color="#5E5E5E">9.3 Reemplazo condicional de valores</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 8</font> 

Mostrar las filas correspondientes a los jugadores del `FC Barcelona` y sustituir los valores del resto con la cadena de texto ` '-----'`.

In [ ]:
df_fifa.where(df_fifa['Club'] == 'FC Barcelona', '-----')

<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section10"></a>
## <font color="#5E5E5E" size=5> 10. Agrupaciones </font>


<a id="section101"></a> 
### <font color="#5E5E5E">10.1 Agrupación por categorías mediante `DataFrame.groupby()` </font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 9</font> 

Mostrar el número de jugadores de cada nacionalidad ordenado de mayor a menor.

In [ ]:
df_fifa.groupby('Nationality').size().sort_values(ascending=False)

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 10</font>

Agrupar los jugadores por nacionalidad y posición, y ordenar los grupos por salario medio de forma descendiente.

In [ ]:
df_fifa.groupby(['Nationality','Position'])['Wage'].mean().sort_values(ascending=False)

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 11</font> 

Repetir el ejercicio anterior, pero de modo que la nacionalidad y la posición sean columnas en el *DataFrame* resultante.

In [ ]:
df_fifa.groupby(['Nationality','Position'], as_index=False)['Wage'].mean().sort_values(by=['Wage'], ascending=False)

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 12</font> 

Mostrar el valor medio de los jugadores cuya valoración media está por encima de 70 y el de los que están por debajo.

In [ ]:
df_fifa.groupby((df_fifa['Overall']>70))['Value'].mean()

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 13</font> 

Mostrar la media de todas las columnas para los jugadores cuya valoración media está por encima de 70 y el de los que están por debajo.

In [ ]:
df_fifa.groupby((df_fifa['Overall']>70)).mean(numeric_only=True)

---

<a id="section107"></a> 
### <font color="#5E5E5E">10.7 Agrupación por categorías externas. Grupos a partir de datos numéricos.</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 14</font> 

Crear categorías de jugador según su valoración media (`Overall`) esté: en el 25% más bajo (`Muy malo`); entre el 25% y el 50% (`Malo`); entre el 50% y el 75% (`Medio`); entre el 75% y el 95% más alto (`Bueno`); y en el 5% más alto (`Top`). Mostrar el número de jugadores y el valor (`Overall`) medio que caen dentro de cada categoría.

In [ ]:
(df_fifa.groupby(pd.qcut(df_fifa['Overall'].dropna(),
                        q=[0, 0.25, 0.5, 0.75, 0.95, 1], 
                        labels=['Muy Malo', 'Malo','Medio','Bueno','Top']), observed=True)
                    ['Overall']
                    .agg([len, 'mean']))

<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section11"></a> 
## <font color="#5E5E5E">11 Operaciones sobre grupos</font>

<a id="section111"></a> 
### <font color="#5E5E5E">11.1 Agregación en grupos.</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 15</font> 

Mostrar los valores máximo y mínimo de `Overall` y `Wage` para cada posición.

In [ ]:
df_fifa.groupby(['Position'], as_index=False)[['Overall','Wage']].agg(['max', 'min'])



<a id="section112"></a> 
### <font color="#5E5E5E">11.2 Filtrado por grupos</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 16</font> 

Crear un *DataFrame* con las filas correspondientes a los clubs en los que el salario (`Wage`) medio está por encima de la media de los salarios.

In [ ]:
mean_wage = df_fifa['Wage'].mean()
df_fifa.groupby('Club').filter(lambda df: df['Wage'].mean()>mean_wage)

In [ ]:
# Similar 
# df_fifa.groupby('Club').filter(lambda df: df['Wage'].mean()>df_fifa['Wage'].mean())

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 17</font> 

Filtrar los datos de modo que se conserven solamente los jugadores que pertenezcan a clubes donde la valoración media es mayor que 75, y que pertenezcan a países para los que hay más de 10 jugadores en estos clubs. Almacenar el resultado en `df_fifa`.

In [ ]:
df_fifa = (df_fifa.groupby('Club').filter(lambda df: df['Overall'].mean()>75)
                  .groupby('Nationality').filter(lambda df: len(df)>10))

df_fifa


<a id="section113"></a> 
### <font color="#5E5E5E">11.3 Transformación por grupo</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 18</font> 

Mostar, para cada jugador (`ID`) qué porcentaje supone su salario sobre el total de salarios (`Wage`) de su club.

In [ ]:
df_fifa.groupby('Club')['Wage'].transform(lambda s_wage: s_wage / s_wage.sum())

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 19</font> 

Repetir el ejercicio anterior, pero utilizar `DataFrame.assign()` para añadir el resultado a una columna denominada `df_fifa['value_club']`. Hacer el ejercicio también sin utilizar `DataFrame.assign()`.

In [ ]:
df_fifa.assign(value_club = df_fifa.groupby('Club')['Overall'].transform(lambda s_wage: s_wage / s_wage.sum())) 

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 20</font> 

Repetir el ejercicio anterior, ordenando el resultado de manera descendente según el valor de la nueva columna, y mostrando solamente los jugadores cuya nacionalidad es española.

In [ ]:
(df_fifa.assign(value_club = df_fifa.groupby('Club')['Overall'].transform(lambda s_wage: s_wage / s_wage.sum())) 
                                    .sort_values(by='value_club', ascending=False) 
                                    .query("Nationality == 'Spain'"))



<a id="section114"></a> 
### <font color="#5E5E5E"> 11.4 `DataFrameGroupBy.apply()`</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 21</font> 

Mostrar, para cada club, el valor del jugador con mayor valoración. Ordenar el resultado de manera descendiente. Repetir el ejercicio pero utilizando `DataFrameGroupBy.apply()`.

In [ ]:
df_fifa.groupby('Club')['Overall'].max().sort_values(ascending=False)

In [ ]:
#df_fifa.groupby('Club').apply(lambda df: df['Overall'].max()).sort_values(ascending=False)
df_fifa.groupby('Club').apply(lambda df: df['Overall'].max(), include_groups=False).sort_values(ascending=False)

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 22</font> 

Repetir el ejercicio anterior, pero mostrando la fila completa correspondiente al jugador.

In [ ]:
df_fifa.groupby('Club', ).apply(lambda df: df.loc[df['Overall'].idxmax()], include_groups=False).sort_values(by='Overall', ascending=False)

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 23</font> 

Repetir el ejercicio anterior pero mostrando, para cada equipo, el jugador con mayor y menor valoración media.

In [ ]:
df_fifa.groupby('Club').apply(lambda df: df.loc[[df['Overall'].idxmax(), df['Overall'].idxmin()]], include_groups=False)

<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section13"></a>
## <font color="#5E5E5E"> 13. Combinación de *Series* y *DataFrames* </font>



<a id="section132"></a> 
### <font color="#5E5E5E">13.2 Concatenación de *Series* y *DataFrames*</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 24</font> 

Concatenar las columnas de los *DataFrame* `df_fifa` y `df_fifa_att` de modo que el *DataFrame* resutante contenga las 200 filas correspondientes a los jugadores que aparecen en `df_fifa`. Utilizar para ello `pd.concat()`.

In [ ]:
pd.concat([df_fifa, df_fifa_att], axis=1, join='inner')



<a id="section133"></a> 
### <font color="#5E5E5E">13.3 Fusión de  *DataFrames*</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 25</font> 

Fundir las filas de los *DataFrames* `df_fifa` y `df_fifa_att` utilizando como clave de unión el índice y de modo que el *DataFrame* resultante contenga los índices del *DataFrame* `df_fifa`. Utilizar ahora `DataFrame.merge()`. Almacenar el resultado en `df_fifa`.

In [ ]:
df_fifa = df_fifa.merge(df_fifa_att, left_index=True, right_index=True, how='left')

<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section14"></a>
## <font color="#5E5E5E"> 14. Índices de varios niveles</font>


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 26</font> 

Establecer como índice principal de `df_fifa` la columna `Club` y como secundario el `ID` del jugador. Para ello, hay que resetear perimero el índice. 

In [ ]:
df_fifa = df_fifa.reset_index().set_index(['Club','ID'])

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 27</font> 

Acceder a los jugadores del Liverpool y al jugador `203376` de ese equipo.

In [ ]:
df_fifa.loc['Liverpool']

In [ ]:
df_fifa.loc[('Liverpool', 203376)]

In [ ]:
df_fifa.xs(203376, level=1)

In [ ]:
df_fifa.reset_index(level=0, inplace=True)

<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section15"></a>
## <font color="#5E5E5E"> 15. Tablas dinámicas y reestructuración </font>

<a id="section151"></a> 
### <font color="#5E5E5E">15.1 Creación de tablas dinámicas</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 28</font> 

Crear una tabla pivot en la que aparezca la media de valoración y salario por equipo y nacionalidad.

In [ ]:
df_fifa.pivot_table(index='Club', columns='Nationality', values=['Overall', 'Wage'], aggfunc=['mean'])




<a id="section52"></a> 
### <font color="#5E5E5E">15.2 Reestructuración de tablas</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 29</font> 

Crear un *DataFrame* en el que la primera columna sea el club del jugador, la segunda el nombre, la tercera se denomine `Concept` y pueda tomar los valores `Overall` o `Potential`, y la cuarta se denomine `Score` y contenga la cantidad correspondiente al concepto reflejado en la columna anterior. 

In [ ]:
df_fifa.melt(id_vars=['Club','Name'], value_vars=['Overall', 'Potential'], var_name='Concept', value_name='Score')

<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>